<a name="top"></a><img src="images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# 模块 2.2：组合逻辑
**上一步：[你的第一个 Chisel 模块](2.1_first_module.ipynb)**<br>
**下一步：[控制流](2.3_control_flow.ipynb)**

## 动机
在本节中，您将了解如何使用 Chisel 组件来实现组合逻辑。
我们将演示三种基本的 Chisel 类型：`UInt` - 无符号整数；`SInt` - 有符号整数，以及 `Bool` - 真或假，如何连接和操作它们。
请注意所有 Chisel 变量是如何声明为 Scala `val` 的。
切勿将 Scala `var` 用于硬件构造，因为构造本身一旦定义就无法更改；只有在运行硬件时其值才能更改。
连线可用于参数化类型。

## 设置

In [ ]:
val path = System.getProperty("user.dir") + "/source/load-ivy.sc"
interp.load.module(ammonite.ops.Path(java.nio.file.FileSystems.getDefault().getPath(path)))

In [ ]:
import chisel3._
import chisel3.util._
import chisel3.tester._
import chisel3.tester.RawTester.test

---
# 常用运算符
既然您已经了解了 `Module` 是如何构造的，那么让我们来创建一些硬件吧！请看下面这个空模块。

In [ ]:
class MyModule extends Module {
  val io = IO(new Bundle {
    val in  = Input(UInt(4.W))
    val out = Output(UInt(4.W))
  })
}

我们将类命名为 `MyModule`，它扩展了 `Module`。这意味着它在 Verilog 中映射到一个硬件模块。我们的 `MyModule` 模块有一个输入和一个输出。输入是一个 4 位无符号整数 (`UInt`)，输出也是如此。

<span style="color:blue">**示例：Scala 和 Chisel 运算符看起来相同**</span><br>
让我们看看我们可以对数据执行哪些不同的操作。

In [ ]:
class MyModule extends Module {
  val io = IO(new Bundle {
    val in  = Input(UInt(4.W))
    val out = Output(UInt(4.W))
  })

  val two  = 1 + 1
  println(two)
  val utwo = 1.U + 1.U
  println(utwo)
  
  io.out := io.in
}
println(getVerilog(new MyModule))

我们创建了两个 `val`。第一个将两个 Scala `Int` 相加，因此 `println` 打印出整数 2。第二个 `val` 将两个 *Chisel* `UInt` 相加，因此 `println` 将其视为硬件节点并打印出类型名称和指针 (`chisel3.core.UInt@d`)。请注意，`1.U` 是从 Scala `Int` (1) 到 Chisel `UInt` 字面量的类型转换。

我们需要将输出驱动到某个地方，因此我们暂时将其连接到输入，就像上一个教程中的直通模块一样。

<span style="color:blue">**示例：不兼容的操作**</span><br>
如果我们将 Chisel `1.U` 与字面量 `1` 相加会发生什么？这些类型是不兼容的，因为前者是值为 1 的硬件线，而后者是值为 1 的 Scala 值。因此 Chisel 会给出类型不匹配错误。

In [ ]:
class MyModuleTwo extends Module {
  val io = IO(new Bundle {
    val in  = Input(UInt(4.W))
    val out = Output(UInt(4.W))
  })

  val twotwo = 1.U + 1
  println(twotwo)
  
  io.out := io.in
}
println(getVerilog(new MyModuleTwo))

在执行操作时，记住类型之间的区别非常重要。Scala 是一种强类型语言，因此任何类型转换都必须是显式的。

<span style="color:blue">**示例：更多 Chisel 运算符**</span><br>
其他常见的操作是减法和乘法。这些操作在无符号整数上按预期处理。让我们看看它们的实际效果。我们展示了 Verilog 代码，尽管有一些底层的 Chisel 特性混淆了我们期望的简单代码。

In [ ]:
class MyOperators extends Module {
  val io = IO(new Bundle {
    val in      = Input(UInt(4.W))
    val out_add = Output(UInt(4.W))
    val out_sub = Output(UInt(4.W))
    val out_mul = Output(UInt(4.W))
  })

  io.out_add := 1.U + 4.U
  io.out_sub := 2.U - 1.U
  io.out_mul := 4.U * 2.U
}
println(getVerilog(new MyOperators))

这是上述操作的一个示例测试程序。我们将创建一个显式的测试程序类，而不是像上一个教程中那样使用匿名测试程序类。这只是编写测试程序的另一种方式。

In [ ]:
test(new MyOperators) {c =>
  c.io.out_add.expect(5.U)
  c.io.out_sub.expect(1.U)
  c.io.out_mul.expect(8.U)
}
println("成功！！")

<span style="color:blue">**示例：Mux 和串联**</span><br>
除了加法、减法和乘法之外，Chisel 还有多路选择器 (mux) 和串联运算符。如下所示。`Mux` 的操作类似于传统的三元运算符，顺序为 (选择信号, 真值时的值, 假值时的值)。请注意，`true.B` 和 `false.B` 是创建 Chisel 布尔字面量的首选方法。`Cat` 的顺序是 MSB 然后是 LSB (其中 B 指的是位)，并且只接受两个参数。串联两个以上的值需要多次调用 `Cat` 或后续章节中介绍的高级 Chisel 和 Scala 功能。

In [ ]:
class MyOperatorsTwo extends Module {
  val io = IO(new Bundle {
    val in      = Input(UInt(4.W))
    val out_mux = Output(UInt(4.W))
    val out_cat = Output(UInt(4.W))
  })

  val s = true.B
  io.out_mux := Mux(s, 3.U, 0.U) // 应该返回 3.U，因为 s 为真
  io.out_cat := Cat(2.U, 1.U)    // 将 2 (b10) 与 1 (b1) 连接起来得到 5 (101)
}

println(getVerilog(new MyOperatorsTwo))

test(new MyOperatorsTwo) { c =>
  c.io.out_mux.expect(3.U)
  c.io.out_cat.expect(5.U)
}
println("成功！！")

请注意 Verilog 如何包含常量而不是实际的 mux 或串联逻辑。这是因为 FIRRTL 转换简化了电路，消除了明显的逻辑。

有关 Chisel 运算符的更完整列表，请参阅 [Chisel 速查表](https://github.com/freechipsproject/chisel-cheatsheet/releases/latest/download/chisel_cheatsheet.pdf)。有关运算符及其实现细节的最完整列表，请查看 [Chisel API](https://chisel-lang.org/api/latest/)。

---
# 练习
要完成这些练习，您可能需要查阅 [Chisel 速查表](https://github.com/freechipsproject/chisel-cheatsheet/releases/latest/download/chisel_cheatsheet.pdf)。

<span style="color:red">**练习：MAC**</span><br>
创建一个 Chisel 模块来实现乘加累积函数 `(A*B)+C`，并通过测试平台。

In [ ]:
class MAC extends Module {
  val io = IO(new Bundle {
    val in_a = Input(UInt(4.W))
    val in_b = Input(UInt(4.W))
    val in_c = Input(UInt(4.W))
    val out  = Output(UInt(8.W))
  })

  ???
}

test(new MAC) { c =>
  val cycles = 100
  import scala.util.Random
  for (i <- 0 until cycles) {
    val in_a = Random.nextInt(16)
    val in_b = Random.nextInt(16)
    val in_c = Random.nextInt(16)
    c.io.in_a.poke(in_a.U)
    c.io.in_b.poke(in_b.U)
    c.io.in_c.poke(in_c.U)
    c.io.out.expect((in_a * in_b + in_c).U)
  }
}
println("成功！！")

<div id="container"><section id="accordion"><div>
<input type="checkbox" id="check-1" />
<label for="check-1"><strong>解决方案</strong></label>
<article>
<pre style="background-color:#f7f7f7">
class MAC extends Module {
  val io = IO(new Bundle {
    val in_a = Input(UInt(4.W))
    val in_b = Input(UInt(4.W))
    val in_c = Input(UInt(4.W))
    val out  = Output(UInt(8.W))
  })

  io.out := (io.in_a * io.in_b) + io.in_c
}
</pre></article></div></section></div>

<span style="color:red">**练习：仲裁器**</span><br>
以下电路将来自 FIFO 的数据仲裁到两个并行处理单元。FIFO 和处理元件 (PE) 通过就绪-有效接口进行通信。构建仲裁器以将数据发送到准备好接收数据的任何 PE，如果两个 PE 都准备好接收数据，则优先处理 PE0。请记住，当至少一个 PE 可以接收数据时，仲裁器应告知 FIFO 它已准备好接收数据。另外，在断言数据有效之前，请等待 PE 断言它已准备就绪。您可能需要二进制运算符来完成此练习。

<img src="images/arbiter.png" width="687" height="177">

In [ ]:
class Arbiter extends Module {
  val io = IO(new Bundle {
    // FIFO
    val fifo_valid = Input(Bool())
    val fifo_ready = Output(Bool())
    val fifo_data  = Input(UInt(16.W))
    
    // PE0
    val pe0_valid  = Output(Bool())
    val pe0_ready  = Input(Bool())
    val pe0_data   = Output(UInt(16.W))
    
    // PE1
    val pe1_valid  = Output(Bool())
    val pe1_ready  = Input(Bool())
    val pe1_data   = Output(UInt(16.W))
  })

  ???
}

test(new Arbiter) { c =>
  import scala.util.Random
  val data = Random.nextInt(65536)
  c.io.fifo_data.poke(data.U)
  
  for (i <- 0 until 8) {
    c.io.fifo_valid.poke((((i >> 0) % 2) != 0).B)
    c.io.pe0_ready.poke((((i >> 1) % 2) != 0).B)
    c.io.pe1_ready.poke((((i >> 2) % 2) != 0).B)

    c.io.fifo_ready.expect((i > 1).B)
    c.io.pe0_valid.expect((i == 3 || i == 7).B)
    c.io.pe1_valid.expect((i == 5).B)
    
    if (i == 3 || i ==7) {
      c.io.pe0_data.expect((data).U)
    } else if (i == 5) {
      c.io.pe1_data.expect((data).U)
    }
  }
}
println("成功！！")

<div id="container"><section id="accordion"><div>
<input type="checkbox" id="check-2" />
<label for="check-2"><strong>解决方案</strong></label>
<article>
<pre style="background-color:#f7f7f7">
  io.fifo_ready := io.pe0_ready || io.pe1_ready
  io.pe0_valid := io.fifo_valid && io.pe0_ready
  io.pe1_valid := io.fifo_valid && io.pe1_ready && !io.pe0_ready
  io.pe0_data := io.fifo_data
  io.pe1_data := io.fifo_data
</pre></article></div></section></div>

<span style="color:red">**练习：参数化加法器（可选）**</span><br>
这个可选练习向您展示了 Chisel 最强大的功能之一，即其参数化能力。为了演示这一点，我们将构建一个参数化加法器，它可以在发生溢出时饱和输出，或者截断结果（即环绕）。

首先，请看下面的 `Module`。我们传递给它的参数称为 `saturate`，类型为 *Scala* `Boolean`。这不是 Chisel `Bool`。因此，我们不是在创建一个可以饱和或截断的单个硬件加法器，而是在创建一个*生成器*，它可以生成饱和硬件加法器*或*截断硬件加法器。这个决定是在编译时做出的。

其次，请注意输入和输出都是 4 位 `UInt`。Chisel具有内置的位宽推断功能，如果您查看[速查表](https://github.com/freechipsproject/chisel-cheatsheet/releases/latest/download/chisel_cheatsheet.pdf)，您会发现正常求和的位宽等于两个输入的最大位宽。这意味着

```scala
val sum = io.in_a + io.in_b
```

将使 `sum` 成为一个 4 位线，其值将是 4 位输入的截断结果。要检查求和是否应该饱和，您需要将结果放在一个 5 位线中。这可以使用 `+&` 求和来完成，如速查表中所示。

```scala
val sum = io.in_a +& io.in_b
```

最后，请注意，将 4 位 `UInt` 线连接到 5 位 `UInt` 线默认会截断 MSB。您可以利用这一点轻松截断非饱和加法器的 5 位和。

In [ ]:
class ParameterizedAdder(saturate: Boolean) extends Module {
  val io = IO(new Bundle {
    val in_a = Input(UInt(4.W))
    val in_b = Input(UInt(4.W))
    val out  = Output(UInt(4.W))
  })

  ???
}

for (saturate <- Seq(true, false)) {
  test(new ParameterizedAdder(saturate)) { c =>
    // 100 次随机测试
    val cycles = 100
    import scala.util.Random
    import scala.math.min
    for (i <- 0 until cycles) {
      val in_a = Random.nextInt(16)
      val in_b = Random.nextInt(16)
      c.io.in_a.poke(in_a.U)
      c.io.in_b.poke(in_b.U)
      if (saturate) {
        c.io.out.expect(min(in_a + in_b, 15).U)
      } else {
        c.io.out.expect(((in_a + in_b) % 16).U)
      }
    }
    
    // 确保我们测试饱和与截断
    c.io.in_a.poke(15.U)
    c.io.in_b.poke(15.U)
    if (saturate) {
      c.io.out.expect(15.U)
    } else {
      c.io.out.expect(14.U)
    }
  }
}
println("成功！！")

<div id="container"><section id="accordion"><div>
<input type="checkbox" id="check-3" />
<label for="check-3"><strong>解决方案</strong></label>
<article>
<pre style="background-color:#f7f7f7">
  val sum = io.in_a +& io.in_b
  if (saturate) {
    io.out := Mux(sum > 15.U, 15.U, sum)
  } else {
    io.out := sum
  }
</pre></article></div></section></div>

---
# 您已完成！

[返回顶部。](#top)